# Nuclear Waste Canister Temperature Prediction — KNN Ensemble
**CIVIL-226 - Introduction to Machine Learning for Engineers — EPFL**  
**Team name:** [To fill]  
**Members:** NAJA Nour | CLERICI Alessandro

## Objective
Predict temperature around nuclear waste canisters using a **KNN ensemble**:
- Two KNN models trained separately on Buffer (x < 1.4m) and OPA (x ≥ 1.4m) zones
- Soft blending at the boundary to avoid discontinuities
- Only Rep 2 (radioactive decay) used for training, matching the test set power profile

## 1. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

np.random.seed(42)
print('Imports OK')

## 2. Load Data

In [ ]:
train   = pd.read_parquet('data/train.parquet')
test    = pd.read_parquet('data/test.parquet')
sensors = pd.read_parquet('data/sensors.parquet')

# Remove duplicate sensors (N206, N213 appear twice with identical coordinates)
n_before = len(sensors)
sensors = sensors.drop_duplicates(subset='sensor', keep='first').reset_index(drop=True)
print(f'Duplicate sensors removed: {n_before - len(sensors)}')
print(f'Sensors : {sensors.shape}  ->  {sensors.columns.tolist()}')
print(f'Train   : {train.shape}   ->  {train.columns.tolist()}')
print(f'Test    : {test.shape}    ->  {test.columns.tolist()}')

## 3. Merge Sensor Positions

In [ ]:
train_full = train.merge(sensors, on='sensor', how='left')
print(train_full.head())
print(train_full.shape)

## 4. Clean Missing Values

In [ ]:
print(f'Rows before NaN removal: {len(train_full)}')
train_full = train_full.dropna(subset=['temperature']).copy()
print(f'Rows after NaN removal : {len(train_full)}')

## 5. Data Exploration

In [ ]:
print(train_full['temperature'].describe(percentiles=[0.001, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 0.999]))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(train_full['temperature'].dropna(), bins=100, color='steelblue', edgecolor='white')
axes[0].set_title('Temperature distribution')
axes[0].set_xlabel('Temperature [°C]')

temp_by_time = train_full.groupby('time')['temperature'].mean()
axes[1].plot(temp_by_time.index / (365.25*24*3600), temp_by_time.values, color='tomato', linewidth=0.8)
axes[1].set_title('Mean temperature over time')
axes[1].set_xlabel('Time [years]')
axes[1].set_ylabel('Mean temperature [°C]')
plt.tight_layout()
plt.show()

## 6. Outlier Removal

Two complementary methods:
- **Global MAD**: catches physically impossible temperatures
- **Local IQR per timestep**: catches values deviating too much from the distribution at a given moment

In [ ]:
# Global outliers via MAD
temp = train_full['temperature']
median_temp = temp.median()
mad_temp    = np.median(np.abs(temp - median_temp))
robust_z    = 0.6745 * (temp - median_temp) / (mad_temp + 1e-8)
train_full['is_global_outlier'] = robust_z.abs() > 6
print(f'Global outliers (MAD): {train_full["is_global_outlier"].sum()}')

# Local outliers per timestep via IQR
time_stats = (
    train_full.groupby('time')['temperature']
    .agg(time_median='median',
         time_q1=lambda x: x.quantile(0.25),
         time_q3=lambda x: x.quantile(0.75))
    .reset_index()
)
time_stats['time_iqr'] = time_stats['time_q3'] - time_stats['time_q1']
train_full = train_full.merge(time_stats, on='time', how='left')
train_full['is_local_outlier'] = (
    (train_full['temperature'] < train_full['time_median'] - 4.0 * train_full['time_iqr']) |
    (train_full['temperature'] > train_full['time_median'] + 4.0 * train_full['time_iqr'])
)
print(f'Local outliers (IQR per timestep): {train_full["is_local_outlier"].sum()}')

# Remove
before_rows = len(train_full)
train_full  = train_full[
    (~train_full['is_global_outlier']) & (~train_full['is_local_outlier'])
].copy()
print(f'Rows removed: {before_rows - len(train_full)} | Remaining: {len(train_full)}')

cols_to_drop = ['is_global_outlier', 'is_local_outlier',
                'time_median', 'time_q1', 'time_q3', 'time_iqr']
train_full = train_full.drop(columns=[c for c in cols_to_drop if c in train_full.columns])

## 7. Sensor Drift Detection & Correction

In [ ]:
time_median = (
    train_full.groupby('time')['temperature']
    .median().rename('global_time_median').reset_index()
)
drift_df = train_full.merge(time_median, on='time', how='left').copy()
drift_df['temp_residual'] = drift_df['temperature'] - drift_df['global_time_median']

t_min = drift_df['time'].min()
t_max = drift_df['time'].max()
drift_df['time_norm_for_drift'] = (drift_df['time'] - t_min) / (t_max - t_min + 1e-8)

drift_records = []
for sensor_id, g in drift_df.groupby('sensor'):
    g = g.sort_values('time_norm_for_drift')
    if len(g) < 20:
        continue
    x = g['time_norm_for_drift'].values
    y = g['temp_residual'].values
    slope = np.polyfit(x, y, 1)[0] if np.std(y) > 1e-8 else 0.0
    corr  = np.corrcoef(x, y)[0, 1] if np.std(y) > 1e-8 else 0.0
    drift_records.append({'sensor': sensor_id, 'drift_slope': slope, 'drift_corr': corr})

sensor_drift   = pd.DataFrame(drift_records)
slope_abs      = sensor_drift['drift_slope'].abs()
slope_threshold = slope_abs.median() + 4 * (np.median(np.abs(slope_abs - slope_abs.median())) + 1e-8)
sensor_drift['is_drift_sensor'] = (
    (sensor_drift['drift_slope'].abs() > slope_threshold) &
    (sensor_drift['drift_corr'].abs() > 0.5)
)
drift_sensors = sensor_drift.loc[sensor_drift['is_drift_sensor'], 'sensor'].unique()
print(f'Drift sensors detected: {len(drift_sensors)}')

# Apply correction
t_min = train_full['time'].min()
t_max = train_full['time'].max()
train_full['time_norm_for_drift'] = (train_full['time'] - t_min) / (t_max - t_min + 1e-8)
drift_map = sensor_drift.set_index('sensor')['drift_slope'].to_dict()
train_full['drift_correction'] = 0.0
for s in drift_sensors:
    mask = train_full['sensor'] == s
    train_full.loc[mask, 'drift_correction'] = (
        drift_map[s] * (train_full.loc[mask, 'time_norm_for_drift'] - 0.5)
    )
train_full['temperature'] = train_full['temperature'] - train_full['drift_correction']
train_full = train_full.drop(columns=['time_norm_for_drift', 'drift_correction'])
print('Drift correction applied.')

## 8. Identify Replication & Filter Rep 2

Each (sensor, timestep) has 3 rows — one per experimental replication:
- **Rep 0**: 500W → 0W (drops around yr 100)
- **Rep 1**: 1400W → 0W (drops around yr 110)
- **Rep 2**: radioactive decay 1488W → 299W over 250 years

The test set matches **Rep 2 exclusively**, so we train only on Rep 2.

In [ ]:
train_full['rep'] = (
    train_full.groupby(['sensor', 'time'])['power']
    .rank(method='first').astype(int) - 1
)
for rep in [0, 1, 2]:
    sub = train_full[train_full['rep'] == rep]
    print(f'Rep {rep}: {sub.sensor.nunique()} sensors, {len(sub):,} rows')

train_full = train_full[train_full['rep'] == 2].drop(columns=['rep']).copy()
print(f'\nAfter filtering Rep 2: {len(train_full):,} rows, {train_full.sensor.nunique()} sensors')

## 9. Feature Engineering

In [ ]:
eps = 1e-8

def add_cum_energy(df):
    """Compute cumulative energy (power × dt) and merge back."""
    power_time = (
        df[['time', 'power']]
        .drop_duplicates(subset=['time'])
        .sort_values('time').copy()
    )
    power_time['dt']         = power_time['time'].diff().fillna(0)
    power_time['cum_energy'] = (power_time['power'] * power_time['dt']).cumsum()
    return df.merge(
        power_time[['time', 'cum_energy']].drop_duplicates(subset=['time']),
        on='time', how='left'
    )

def add_features(df, t_max_ref):
    """
    Add physics-motivated engineered features.
    t_max_ref: max time from training data for consistent time_norm on test.
    """
    df = df.copy()
    df['dist2']            = df['coor_x']**2 + df['coor_y']**2
    df['dist_center']      = np.sqrt(df['dist2'])
    df['dist_canister']    = np.sqrt((df['coor_x'] - 0.7)**2 + (df['coor_y'] - 1.2)**2)
    df['is_opa']           = (df['coor_x'] > 1.4).astype(float)
    df['time_log']         = np.log1p(df['time'])
    df['time_norm']        = df['time'] / t_max_ref
    df['power_over_dist2'] = df['power'] / (df['dist2'] + eps)
    df['diffusion']        = df['dist2'] / (df['time'] + eps)
    return df

t_max_ref  = train_full['time'].max()
train_full = add_cum_energy(train_full)
train_full = add_features(train_full, t_max_ref)
print('Features:', train_full.columns.tolist())

## 10. Final Checks

In [ ]:
print('Shape:', train_full.shape)
assert train_full['temperature'].notna().all()
assert np.isfinite(train_full['temperature']).all()
print('All checks passed.')

## 11. Buffer / OPA Zone Split

The buffer (x < 1.4m) and OPA (x ≥ 1.4m) zones have fundamentally different thermal properties.
Training two separate KNN models allows each to specialize on its zone's dynamics.

The boundary threshold x = 1.4m corresponds to the physical interface between the compacted clay buffer and the natural OPA rock.

In [ ]:
BUFFER_THRESHOLD = 1.4

train_buffer = train_full[train_full['coor_x'] <  BUFFER_THRESHOLD].copy()
train_opa    = train_full[train_full['coor_x'] >= BUFFER_THRESHOLD].copy()

print(f'Buffer : {train_buffer.sensor.nunique()} sensors, {len(train_buffer):,} rows')
print(f'OPA    : {train_opa.sensor.nunique()} sensors, {len(train_opa):,} rows')

## 12. Features & Train/Validation Split by Sensor

Split by sensor (not by row) to properly evaluate generalization to unseen sensor positions — the actual test scenario.

Features chosen to keep the KNN distance space clean:
- Spatial: `coor_x`, `coor_y`
- Temporal: `time_log` (single log-scale representation of time)
- Physical: `power`, `power_over_dist2`, `diffusion`

In [ ]:
FEATURES = [
    'coor_x', 'coor_y',
    'time_log',
    'power',
    'power_over_dist2',
    'diffusion',
]
TARGET = 'temperature'

def sensor_split(df, val_ratio=0.2, seed=42):
    """Split by sensor to avoid spatial leakage between train and val."""
    unique_sensors = df['sensor'].unique()
    rng = np.random.default_rng(seed)
    val_sensors   = rng.choice(unique_sensors, size=int(val_ratio * len(unique_sensors)), replace=False)
    train_sensors = np.setdiff1d(unique_sensors, val_sensors)
    train_df = df[df['sensor'].isin(train_sensors)].copy()
    val_df   = df[df['sensor'].isin(val_sensors)].copy()
    assert set(train_df['sensor']).isdisjoint(set(val_df['sensor']))
    return train_df, val_df

train_df_buf, val_df_buf = sensor_split(train_buffer)
train_df_opa, val_df_opa = sensor_split(train_opa)

print(f'Buffer — Train: {train_df_buf.sensor.nunique()} sensors | Val: {val_df_buf.sensor.nunique()} sensors')
print(f'OPA    — Train: {train_df_opa.sensor.nunique()} sensors | Val: {val_df_opa.sensor.nunique()} sensors')

## 13. Normalisation

**Critical for KNN**: distance is computed directly on feature values.
Each zone gets its own scaler fitted only on its training data.

In [ ]:
def fit_and_scale(train_df, val_df, features):
    """Fit StandardScaler on train, apply to train and val."""
    X_train = train_df[features].values
    y_train = train_df[TARGET].values
    X_val   = val_df[features].values
    y_val   = val_df[TARGET].values

    scaler    = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_val_s   = scaler.transform(X_val)

    return X_train_s, y_train, X_val_s, y_val, scaler

X_train_buf, y_train_buf, X_val_buf, y_val_buf, scaler_buf = fit_and_scale(train_df_buf, val_df_buf, FEATURES)
X_train_opa, y_train_opa, X_val_opa, y_val_opa, scaler_opa = fit_and_scale(train_df_opa, val_df_opa, FEATURES)

print(f'Buffer — X_train: {X_train_buf.shape} | X_val: {X_val_buf.shape}')
print(f'OPA    — X_train: {X_train_opa.shape} | X_val: {X_val_opa.shape}')

## 14. Hyperparameter Tuning — Best k per Zone

In [ ]:
def find_best_k(X_train_s, y_train, X_val_s, y_val, k_values, sample_size=100_000, zone_name=''):
    """Search best k on a subsample of training data for speed."""
    idx = np.random.choice(len(X_train_s), min(sample_size, len(X_train_s)), replace=False)
    rmse_k = []
    for k in k_values:
        knn = KNeighborsRegressor(n_neighbors=k, n_jobs=-1, algorithm='ball_tree')
        knn.fit(X_train_s[idx], y_train[idx])
        pred = knn.predict(X_val_s[:10_000])
        rmse = np.sqrt(mean_squared_error(y_val[:10_000], pred))
        rmse_k.append(rmse)
        print(f'  [{zone_name}] k={k:3d} -> RMSE = {rmse:.4f}')
    best_k = k_values[np.argmin(rmse_k)]
    print(f'  => Best k for {zone_name}: {best_k}\n')
    return best_k, rmse_k

k_values = [3, 5, 7, 10]

best_k_buf, rmse_k_buf = find_best_k(X_train_buf, y_train_buf, X_val_buf, y_val_buf, k_values, zone_name='Buffer')
best_k_opa, rmse_k_opa = find_best_k(X_train_opa, y_train_opa, X_val_opa, y_val_opa, k_values, zone_name='OPA')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, rmse_k, best_k, zone in zip(axes,
                                     [rmse_k_buf, rmse_k_opa],
                                     [best_k_buf, best_k_opa],
                                     ['Buffer', 'OPA']):
    ax.plot(k_values, rmse_k, 'o-', color='steelblue')
    ax.axvline(best_k, color='tomato', linestyle='--', label=f'Best k={best_k}')
    ax.set_title(f'k selection — {zone}')
    ax.set_xlabel('k')
    ax.set_ylabel('RMSE (validation sample)')
    ax.legend()
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 15. Train Final KNN Models

In [ ]:
knn_buf = KNeighborsRegressor(n_neighbors=best_k_buf, n_jobs=-1, algorithm='ball_tree')
knn_buf.fit(X_train_buf, y_train_buf)
print(f'KNN Buffer trained — k={best_k_buf}, {len(X_train_buf):,} samples')

knn_opa = KNeighborsRegressor(n_neighbors=best_k_opa, n_jobs=-1, algorithm='ball_tree')
knn_opa.fit(X_train_opa, y_train_opa)
print(f'KNN OPA    trained — k={best_k_opa}, {len(X_train_opa):,} samples')

## 16. Ensemble Prediction with Soft Boundary Blending

At the Buffer/OPA boundary (x ≈ 1.4m), instead of a hard cutoff, we use a **sigmoid blend**:

$$\alpha = \sigma\left(\frac{x - 1.4}{\delta}\right), \quad \hat{y} = (1-\alpha)\cdot\hat{y}_{\text{buffer}} + \alpha\cdot\hat{y}_{\text{OPA}}$$

where δ controls the width of the transition zone. This avoids a discontinuity at x=1.4 (which caused N4 to have RMSE=24 in the single-model approach).

In [ ]:
def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def ensemble_predict(df, features, scaler_buf, scaler_opa, knn_buf, knn_opa,
                     threshold=1.4, blend_width=0.3):
    """
    Ensemble prediction with soft sigmoid blending at the Buffer/OPA boundary.

    Parameters
    ----------
    df          : DataFrame with features and coor_x
    threshold   : Buffer/OPA boundary (x = 1.4m)
    blend_width : controls transition zone width (smaller = sharper transition)
    """
    X = df[features].values

    # Predict with both models (each with its own scaler)
    X_buf_s = scaler_buf.transform(X)
    X_opa_s = scaler_opa.transform(X)

    y_buf = knn_buf.predict(X_buf_s)
    y_opa = knn_opa.predict(X_opa_s)

    # Sigmoid blend weight: 0 = pure buffer, 1 = pure OPA
    alpha = sigmoid((df['coor_x'].values - threshold) / blend_width)

    return (1 - alpha) * y_buf + alpha * y_opa

## 17. Evaluate on Validation Set

In [ ]:
# Combine val sets for global evaluation
val_df_all = pd.concat([val_df_buf, val_df_opa], ignore_index=True)
y_val_all  = val_df_all[TARGET].values

y_val_pred = ensemble_predict(val_df_all, FEATURES, scaler_buf, scaler_opa, knn_buf, knn_opa)

mae  = mean_absolute_error(y_val_all, y_val_pred)
rmse = np.sqrt(mean_squared_error(y_val_all, y_val_pred))
r2   = r2_score(y_val_all, y_val_pred)

print(f'Ensemble Validation MAE  : {mae:.4f} °C')
print(f'Ensemble Validation RMSE : {rmse:.4f} °C')
print(f'Ensemble Validation R²   : {r2:.4f}')

# Per-zone breakdown
for zone, val_df_z, y_val_z in [
    ('Buffer', val_df_buf, y_val_buf),
    ('OPA',    val_df_opa, y_val_opa)
]:
    y_pred_z = ensemble_predict(val_df_z, FEATURES, scaler_buf, scaler_opa, knn_buf, knn_opa)
    rmse_z   = np.sqrt(mean_squared_error(y_val_z, y_pred_z))
    print(f'  {zone} RMSE: {rmse_z:.4f} °C')

In [ ]:
# True vs predicted scatter
plt.figure(figsize=(6, 6))
plt.scatter(y_val_all, y_val_pred, s=3, alpha=0.3)
mn, mx = min(y_val_all.min(), y_val_pred.min()), max(y_val_all.max(), y_val_pred.max())
plt.plot([mn, mx], [mn, mx], 'r--')
plt.xlabel('True temperature [°C]')
plt.ylabel('Predicted temperature [°C]')
plt.title('KNN Ensemble — True vs Predicted on unseen sensors')
plt.grid(True)
plt.tight_layout()
plt.show()

## 18. Error Analysis by Sensor

In [ ]:
val_results = val_df_all.copy()
val_results['y_true']    = y_val_all
val_results['y_pred']    = y_val_pred
val_results['abs_error'] = np.abs(y_val_all - y_val_pred)
val_results['sq_error']  = (y_val_all - y_val_pred)**2

sensor_metrics = (
    val_results
    .groupby('sensor')
    .agg(
        mae  =('abs_error', 'mean'),
        rmse =('sq_error',  lambda x: np.sqrt(np.mean(x))),
        coor_x=('coor_x', 'first'),
        coor_y=('coor_y', 'first'),
    )
    .sort_values('rmse', ascending=False)
)
print('Worst sensors by RMSE:')
print(sensor_metrics.head(10))

In [ ]:
# Spatial error map
plt.figure(figsize=(8, 5))
sc = plt.scatter(
    sensor_metrics['coor_x'],
    sensor_metrics['coor_y'],
    c=sensor_metrics['rmse'],
    cmap='hot_r', s=80
)
plt.colorbar(sc, label='RMSE [°C]')
plt.axvline(1.4, color='blue', linestyle='--', alpha=0.5, label='Buffer/OPA boundary (x=1.4)')
plt.xlabel('coor_x')
plt.ylabel('coor_y')
plt.title('Validation RMSE by sensor position — KNN Ensemble')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 19. Final Predictions & Submission

In [ ]:
# Build test_full with same preprocessing pipeline
test_full = test.merge(sensors, on='sensor', how='left')
test_full = add_cum_energy(test_full)
test_full = add_features(test_full, t_max_ref)

print(f'test_full shape: {test_full.shape}')
print(f'Missing in features: {test_full[FEATURES].isna().sum().sum()}')

In [ ]:
y_pred = ensemble_predict(test_full, FEATURES, scaler_buf, scaler_opa, knn_buf, knn_opa)

submission = pd.DataFrame({
    'Id': np.arange(len(test), dtype=int),
    'temperature': y_pred.astype(float)
})

assert list(submission.columns) == ['Id', 'temperature']
assert len(submission) == len(test)
assert np.isfinite(submission['temperature']).all()
assert submission.isna().sum().sum() == 0

submission.to_csv('submission.csv', index=False)
print(f'submission.csv saved — {len(submission)} rows')
display(submission.head())

In [ ]:
# Sanity check: train vs test predictions over time
test_full['temp_pred'] = y_pred
temp_train = train_full.groupby('time')['temperature'].mean()
temp_pred  = test_full.groupby('time')['temp_pred'].mean()

plt.figure(figsize=(10, 4))
plt.plot(temp_train.index / (365.25*24*3600), temp_train.values,
         label='Train Rep2 (true)', color='tomato', alpha=0.7, linewidth=0.8)
plt.plot(temp_pred.index / (365.25*24*3600), temp_pred.values,
         label='Test (predicted)', color='steelblue', alpha=0.7, linewidth=0.8)
plt.title('Sanity check: Train vs Test predictions over time')
plt.xlabel('Time [years]')
plt.ylabel('Mean temperature [°C]')
plt.legend()
plt.tight_layout()
plt.show()

## 20. Spatial Temperature Map: True vs Predicted

In [ ]:
# Mean temperature per sensor position
train_sensor_temp = (
    train_full
    .groupby('sensor')
    .agg(coor_x=('coor_x', 'first'),
         coor_y=('coor_y', 'first'),
         mean_temp=('temperature', 'mean'))
    .reset_index()
)

test_sensor_temp = (
    test_full
    .groupby('sensor')
    .agg(coor_x=('coor_x', 'first'),
         coor_y=('coor_y', 'first'),
         mean_temp=('temp_pred', 'mean'))
    .reset_index()
)

# Shared color scale for fair comparison
vmin = min(train_sensor_temp['mean_temp'].min(), test_sensor_temp['mean_temp'].min())
vmax = max(train_sensor_temp['mean_temp'].max(), test_sensor_temp['mean_temp'].max())

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sc1 = axes[0].scatter(
    train_sensor_temp['coor_x'], train_sensor_temp['coor_y'],
    c=train_sensor_temp['mean_temp'],
    cmap='hot_r', vmin=vmin, vmax=vmax,
    s=80, edgecolors='gray', linewidths=0.3
)
axes[0].set_title('True temperatures — Train Rep2 sensors', fontsize=13)
axes[0].set_xlabel('coor_x [m]')
axes[0].set_ylabel('coor_y [m]')
axes[0].axvline(1.4, color='blue', linestyle='--', alpha=0.5, label='Buffer/OPA x=1.4')
axes[0].legend(fontsize=9)
axes[0].grid(True, alpha=0.3)
plt.colorbar(sc1, ax=axes[0], label='Mean temperature [°C]')

sc2 = axes[1].scatter(
    test_sensor_temp['coor_x'], test_sensor_temp['coor_y'],
    c=test_sensor_temp['mean_temp'],
    cmap='hot_r', vmin=vmin, vmax=vmax,
    s=80, edgecolors='gray', linewidths=0.3
)
axes[1].set_title('Predicted temperatures — Test sensors (KNN Ensemble)', fontsize=13)
axes[1].set_xlabel('coor_x [m]')
axes[1].set_ylabel('coor_y [m]')
axes[1].axvline(1.4, color='blue', linestyle='--', alpha=0.5, label='Buffer/OPA x=1.4')
axes[1].legend(fontsize=9)
axes[1].grid(True, alpha=0.3)
plt.colorbar(sc2, ax=axes[1], label='Mean temperature [°C]')

plt.suptitle('Spatial temperature distribution: true vs predicted\n'
             'Expected: radial gradient, hot near canister (x≈0), cool far away',
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()